# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [ ]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        # Coordenadas identificadas en la imagen (row, col)
        self.start = (0, 0)
        self.walls = {(0, 3), (1, 1), (2, 4), (4, 2)}
        self.slippery_states = {(1, 2), (2, 1), (3, 3)}

        self.terminal_states = {
            (0, 5): 10.0,  # entrega
            (2, 2):  2.0,  # carga
            (3, 5): -10.0, # peligro mortal
        }

        self.danger_states = {
            (1, 4): -3.0,
            (4, 1): -3.0,
        }

        self.living_reward = -1.0
        self.gamma = 0.9
        self.normal_probs = (0.90, 0.05, 0.05)
        self.slippery_probs = (0.60, 0.20, 0.20)

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        r, c = state
        return 0 <= r < self.height and 0 <= c < self.width and state not in self.walls

    def states(self):
        return [(r, c) for r in range(self.height)
                for c in range(self.width) if (r, c) not in self.walls]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        if state in self.terminal_states:
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        return self.living_reward

    def get_transition_probs(self, state, action):
        """Devuelve una lista (siguiente_estado, probabilidad)."""
        if self.is_terminal(state):
            return [(state, 1.0)]

        # Para (dr, dc), izquierda=(-dc, dr) y derecha=(dc, -dr).
        dr, dc = action
        left = (-dc, dr)
        right = (dc, -dr)
        probs = self.slippery_probs if state in self.slippery_states else self.normal_probs

        accumulated = {}
        for movement, probability in zip((action, left, right), probs):
            candidate = (state[0] + movement[0], state[1] + movement[1])
            next_state = candidate if self.is_valid_state(candidate) else state
            accumulated[next_state] = accumulated.get(next_state, 0.0) + probability

        return list(accumulated.items())



### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [ ]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 26
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [ ]:
def expected_next_value(grid, state, action, V):
    return sum(probability * V[next_state]
               for next_state, probability in grid.get_transition_probs(state, action))


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    V = {state: 0.0 for state in grid.states()}

    for iteration in range(1, max_iter + 1):
        new_V = V.copy()
        delta = 0.0

        for state in grid.states():
            if grid.is_terminal(state):
                new_V[state] = grid.get_reward(state)
            else:
                best_future = max(expected_next_value(grid, state, action, V)
                                  for action in grid.actions)
                new_V[state] = grid.get_reward(state) + grid.gamma * best_future
            delta = max(delta, abs(new_V[state] - V[state]))

        V = new_V
        if delta < threshold:
            return V, iteration

    raise RuntimeError("Value Iteration no convergió dentro de max_iter")


def extract_policy(grid, V):
    return {
        state: max(grid.actions,
                   key=lambda action: expected_next_value(grid, state, action, V))
        for state in grid.states() if not grid.is_terminal(state)
    }


## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [ ]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    V = {state: 0.0 for state in grid.states()}

    for iteration in range(1, max_iter + 1):
        new_V = V.copy()
        delta = 0.0

        for state in grid.states():
            if grid.is_terminal(state):
                new_V[state] = grid.get_reward(state)
            else:
                future = expected_next_value(grid, state, policy[state], V)
                new_V[state] = grid.get_reward(state) + grid.gamma * future
            delta = max(delta, abs(new_V[state] - V[state]))

        V = new_V
        if delta < threshold:
            return V, iteration

    raise RuntimeError("Policy Evaluation no convergió dentro de max_iter")


def policy_improvement(grid, V):
    return extract_policy(grid, V)


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    policy = {state: grid.actions[0] for state in grid.states()
              if not grid.is_terminal(state)}
    history = []

    for improvement in range(1, max_iter + 1):
        V, evaluation_iterations = policy_evaluation(grid, policy, threshold)
        new_policy = policy_improvement(grid, V)
        changes = sum(policy[state] != new_policy[state] for state in policy)
        history.append({"mejora": improvement,
                        "iteraciones_evaluacion": evaluation_iterations,
                        "cambios": changes})

        if changes == 0:
            return policy, V, history
        policy = new_policy

    raise RuntimeError("Policy Iteration no convergió dentro de max_iter")




## Parte 4 — Visualización y comparación


In [ ]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [ ]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")

=== VALUE ITERATION ===
Iteraciones: 20

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

=== POLICY ITERATION ===
Historia: [{'mejora': 1, 'iteraciones_evaluacion': 89, 'cambios': 16}, {'mejora': 2, 'iteraciones_evaluacion': 77, 'cambios': 5}, {'mejora': 3, 'iteraciones_evaluacion': 22, 'cambios': 1}, {'mejora': 4, 'iteraciones_evaluacion': 22, 'cambios': 0}]

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.7


## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?

Con los valores originales, el robot prefiere llegar a la estación de carga +2.

Aunque la primera acción desde START es moverse hacia la derecha, debido a las transiciones aleatorias puede terminar en diferentes terminales. La mayor probabilidad es llegar a la carga +2.

2. ¿Por qué una recompensa menor podría ser óptima?

Porque la carga +2 está más cerca. Para llegar a la entrega +10, el robot debe recorrer una ruta más larga, acumular costos de -1 por cada paso y exponerse a movimientos accidentales.

Además, como gamma = 0.9, las recompensas lejanas tienen menos valor que las recompensas cercanas.

3. ¿En qué estados el piso resbaloso cambia la decisión?

Al comparar la política original con otra donde todo el piso es normal, cambian las decisiones en:

(1,2): con piso resbaloso el robot baja ↓; con piso normal iría a la derecha →.
(3,1): con piso resbaloso va a la derecha →; con piso normal subiría ↑.

Esto demuestra que el piso resbaloso también puede afectar decisiones tomadas desde celdas cercanas.

4. ¿Qué papel cumple el costo por paso `-1`?

El costo -1 castiga al robot por demorarse. Esto hace que prefiera rutas cortas y evita que se quede moviéndose indefinidamente.

También penaliza los intentos que chocan contra una pared o un borde, porque el robot permanece en la misma celda, pero igualmente recibe el costo correspondiente a ese estado.

5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

Porque las probabilidades dependen del tipo de piso del estado actual:

Piso normal: 0.90, 0.05, 0.05.
Piso resbaloso: 0.60, 0.20, 0.20.

Por eso, antes de calcular la transición, se debe comprobar si el estado actual pertenece a slippery_states.


### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.
